# Reclaim.ai SDK Exploration Notebook

This notebook provides a comprehensive guide to the **reclaim-sdk**, covering both **Smart Habits** and **Tasks** with equal depth.

## What You'll Learn
- Authenticate with Reclaim.ai API
- Explore and manage Smart Habits
- Create, update, and track Tasks
- Analyze scheduling patterns and productivity
- Export data to Obsidian, CSV, and JSON

## Prerequisites
- reclaim-sdk installed (`pip install -e /Users/raymondyee/C/src/reclaim-sdk`)
- RECLAIM_TOKEN from https://app.reclaim.ai/settings/developer

---
## Section 1: Setup and Authentication

In [ ]:
# Cell 1: Import libraries
import os
import json
import subprocess

from datetime import datetime, timedelta, timezone
from typing import List, Optional, Dict, Any

# Reclaim SDK imports
from reclaim_sdk.client import ReclaimClient
from reclaim_sdk.resources.task import Task, TaskPriority, TaskStatus, EventCategory, EventColor
from reclaim_sdk.resources.habit import (
    SmartHabit, Habit, SmartHabitPeriod, HabitChangeLogEntry,
    HabitStatus, EventType, ChangeReason, RecurrenceFrequency
)
from reclaim_sdk.resources.hours import Hours
from reclaim_sdk.exceptions import ReclaimAPIError, RecordNotFound

# Display configuration
from IPython.display import display, Markdown
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("Imports successful!")

In [ ]:
# Cell 2: Authentication
# Method 1: Environment variable (recommended)
# Run in terminal first:
#   export RECLAIM_TOKEN=$(op read "op://RY/reclaim.ai 2026.01.08/credential")

# Method 2: Direct configuration (for testing only - don't commit tokens!)
# ReclaimClient.configure(token="your_token_here")

# Verify connection
try:
    # insert your specific method of obtaining the token here
    # for rdhyee's setup, we use 1Password CLI to read the token
    RECLAIM_TOKEN = subprocess.run(["op", "read", "op://RY/reclaim.ai 2026.01.08/credential"], capture_output=True, text=True).stdout.strip()
    client = ReclaimClient()
    client.configure(token=RECLAIM_TOKEN)
    habits = SmartHabit.list(client)
    print(f"Connected! Found {len(habits)} Smart Habits")
except Exception as e:
    print(f"Connection failed: {e}")
    print("Make sure RECLAIM_TOKEN is set in your environment")

### Quick Reference

| Resource | Primary Methods |
|----------|----------------|
| **SmartHabit** | `list()`, `get(id)`, `get_by_title()`, `save()`, `enable()`, `disable()`, `get_changelog()` |
| **Task** | `list()`, `get(id)`, `save()`, `delete()`, `mark_complete()`, `start()`, `stop()`, `log_work()`, `add_time()` |
| **Hours** | `list()` (read-only time schemes) |

---
## Section 2: Smart Habits Deep Dive

Smart Habits are Reclaim's primary habit system. They:
- Automatically schedule blocks on your calendar
- Respect your working hours and existing events
- Track completion via periods
- Can be enabled/disabled without losing configuration

In [ ]:
# Cell 3: List All Habits
habits = SmartHabit.list()

print(f"Total: {len(habits)} habits\n")
print(f"{'Status':<8} {'Title':<30} {'Type':<12} {'Duration':<12} {'Periods'}")
print("-" * 80)

for h in habits:
    status = "[ON]" if h.is_enabled else "[--]"
    duration = f"{h.duration_min}-{h.duration_max}m"
    print(f"{status:<8} {h.title[:28]:<30} {h.event_type.value:<12} {duration:<12} {h.instance_count}")

In [ ]:
# Cell 4: Habits as DataFrame
habit_data = []
for h in habits:
    habit_data.append({
        "ID": h.lineage_id,
        "Title": h.title,
        "Status": h.status.value,
        "Type": h.event_type.value,
        "Category": h.category.value,
        "Min (mins)": h.duration_min,
        "Max (mins)": h.duration_max,
        "Priority": h.priority,
        "Periods": h.instance_count,
        "Recurrence": h.recurrence.frequency.value if h.recurrence else "N/A",
    })

habits_df = pd.DataFrame(habit_data)
display(habits_df)

In [ ]:
# Cell 5: Get Specific Habit by ID
def explore_habit(lineage_id: int):
    """Get detailed information about a specific habit."""
    habit = SmartHabit.get(lineage_id)
    
    print(f"=== {habit.title} ===")
    print(f"ID: {habit.lineage_id}")
    print(f"Status: {habit.status.value} ({'enabled' if habit.is_enabled else 'disabled'})")
    print(f"Type: {habit.event_type.value} ({habit.category.value})")
    print(f"Duration: {habit.duration_min}-{habit.duration_max} minutes")
    print(f"Priority: {habit.priority}")
    print(f"Description: {habit.description or '(none)'}")
    
    if habit.recurrence:
        print(f"\nRecurrence:")
        print(f"  Frequency: {habit.recurrence.frequency.value}")
        print(f"  Ideal Days: {', '.join(habit.recurrence.ideal_days)}")
        print(f"  Interval: {habit.recurrence.interval}")
    
    print(f"\nPeriods: {habit.instance_count} scheduled")
    if habit.next_period:
        np = habit.next_period
        print(f"Next Period: {np.start} at {np.event_start}")
    
    return habit

# Example: explore the first habit
if habits:
    example_habit = explore_habit(habits[0].lineage_id)

In [ ]:
# Cell 6: Search Habits by Title
def search_habits(query: str) -> List[SmartHabit]:
    """Search habits by partial title match (case-insensitive)."""
    habits = SmartHabit.list()
    query_lower = query.lower()
    matches = [h for h in habits if query_lower in h.title.lower()]
    
    print(f"Found {len(matches)} habits matching '{query}':")
    for h in matches:
        status = "ON" if h.is_enabled else "OFF"
        print(f"  [{status}] {h.title} (ID: {h.lineage_id})")
    
    return matches

# Try searching (modify the query)
results = search_habits("walk")

In [ ]:
# Cell 7: View Scheduled Periods
def view_periods(habit: SmartHabit, limit: int = 10):
    """Display scheduled periods for a habit."""
    print(f"=== Periods for {habit.title} ===\n")
    
    periods_data = []
    for p in habit.periods[:limit]:
        periods_data.append({
            "Date": p.start,
            "Start": p.event_start.strftime("%I:%M %p") if p.event_start else "Not scheduled",
            "End": p.event_end.strftime("%I:%M %p") if p.event_end else "-",
            "Done": "Yes" if p.done else "",
            "Locked": "Yes" if p.locked else "",
        })
    
    df = pd.DataFrame(periods_data)
    display(df)

# View periods for the first habit
if habits:
    view_periods(habits[0])

In [ ]:
# Cell 8: View Habit Changelog
def view_changelog(habit: SmartHabit, limit: int = 10):
    """Display recent changelog entries for a habit."""
    changelog = habit.get_changelog(limit=limit)
    
    print(f"=== Changelog for {habit.title} ({len(changelog)} entries) ===\n")
    
    for entry in changelog:
        time_str = entry.changed_at.strftime("%m/%d %I:%M%p")
        reason = entry.reason.value.replace("SMART_SERIES_EVENT_", "")
        
        if entry.is_move and entry.previous_start and entry.new_start:
            prev = entry.previous_start.strftime("%m/%d %I:%M%p")
            new = entry.new_start.strftime("%m/%d %I:%M%p")
            print(f"{time_str}: {reason} - {prev} -> {new}")
        else:
            print(f"{time_str}: {reason}")

# View changelog for the first habit
if habits:
    view_changelog(habits[0])

In [ ]:
# Cell 9: Enable/Disable Habits
def toggle_habit(habit: SmartHabit, enable: bool = None) -> None:
    """Toggle or set habit enabled state."""
    if enable is None:
        enable = not habit.is_enabled
    
    print(f"Before: {habit.title} is {'enabled' if habit.is_enabled else 'disabled'}")
    
    if enable:
        habit.enable()
        print(f"After: ENABLED")
    else:
        habit.disable()
        print(f"After: DISABLED")

# Example (uncomment to use):
# habit = SmartHabit.get_by_title("your habit name")
# toggle_habit(habit, enable=False)  # Disable
# toggle_habit(habit, enable=True)   # Enable

In [ ]:
# Cell 10: Update Habit Properties
def update_habit(habit: SmartHabit, **kwargs) -> SmartHabit:
    """Update habit properties and save."""
    original_values = {}
    
    for key, value in kwargs.items():
        if hasattr(habit, key):
            original_values[key] = getattr(habit, key)
            setattr(habit, key, value)
            print(f"  {key}: {original_values[key]} -> {value}")
    
    habit.save()
    print(f"\nSaved changes to {habit.title}")
    return habit

# Example (uncomment to use):
# habit = SmartHabit.get_by_title("your habit name")
# update_habit(habit, 
#     duration_min=30,
#     duration_max=45,
#     description="Updated via notebook"
# )

In [ ]:
# Cell 11: Filter Habits
def filter_habits(
    status: Optional[HabitStatus] = None,
    category: Optional[str] = None,  # "WORK" or "PERSONAL"
    event_type: Optional[EventType] = None,
    min_duration: Optional[int] = None,
    has_periods: bool = None,
) -> List[SmartHabit]:
    """Filter habits by multiple criteria."""
    habits = SmartHabit.list()
    
    if status:
        habits = [h for h in habits if h.status == status]
    if category:
        habits = [h for h in habits if h.category.value == category]
    if event_type:
        habits = [h for h in habits if h.event_type == event_type]
    if min_duration:
        habits = [h for h in habits if h.duration_min >= min_duration]
    if has_periods is not None:
        if has_periods:
            habits = [h for h in habits if h.instance_count > 0]
        else:
            habits = [h for h in habits if h.instance_count == 0]
    
    return habits

# Examples
active_habits = filter_habits(status=HabitStatus.ACTIVE)
work_habits = filter_habits(category="WORK")
personal_with_periods = filter_habits(category="PERSONAL", has_periods=True)

print(f"Active: {len(active_habits)}, Work: {len(work_habits)}, Personal w/periods: {len(personal_with_periods)}")

### Smart Habit Summary

| Action | Method | Notes |
|--------|--------|-------|
| List all | `SmartHabit.list()` | Returns all habits |
| Get by ID | `SmartHabit.get(lineage_id)` | Primary identifier |
| Search by title | `SmartHabit.get_by_title("...")` | Case-insensitive partial match |
| Update | `habit.property = ...; habit.save()` | Persists to server |
| Enable | `habit.enable()` | Activates scheduling |
| Disable | `habit.disable()` | Pauses without deleting |
| Changelog | `habit.get_changelog(limit=N)` | Scheduling history |

---
## Section 3: Tasks Deep Dive

Tasks in Reclaim are one-time items with:
- Due dates and time estimates
- Priority levels (P1-P4)
- Status tracking (NEW, SCHEDULED, IN_PROGRESS, COMPLETE)
- Planner actions (start, stop, log work, prioritize)
- Time chunk management (15-minute increments)

In [ ]:
# Cell 12: List All Tasks
tasks = Task.list()

print(f"Total: {len(tasks)} tasks\n")

# Group by status
by_status = {}
for t in tasks:
    status = t.status.value if t.status else "UNKNOWN"
    by_status.setdefault(status, []).append(t)

for status, task_list in sorted(by_status.items()):
    print(f"\n=== {status} ({len(task_list)}) ===")
    for t in task_list[:5]:  # Show first 5 of each
        due_str = t.due.strftime("%m/%d") if t.due else "No due date"
        duration = f"{t.duration}h" if t.duration else "?"
        priority = t.priority.value if t.priority else "?"
        print(f"  [{priority}] {t.title[:40]:<40} {due_str} ({duration})")

In [ ]:
# Cell 13: Tasks DataFrame
task_data = []
for t in tasks:
    task_data.append({
        "ID": t.id,
        "Title": t.title[:40] if t.title else "",
        "Status": t.status.value if t.status else "UNKNOWN",
        "Priority": t.priority.value if t.priority else "?",
        "Due": t.due.strftime("%Y-%m-%d") if t.due else None,
        "Duration (h)": t.duration,
        "Spent (h)": t.time_chunks_spent / 4 if t.time_chunks_spent else 0,
        "Remaining (h)": t.time_chunks_remaining / 4 if t.time_chunks_remaining else None,
        "On Deck": "Yes" if t.on_deck else "",
        "At Risk": "Yes" if t.at_risk else "",
        "Category": t.event_category.value if t.event_category else "?",
    })

tasks_df = pd.DataFrame(task_data)
display(tasks_df)

In [ ]:
# Cell 14: Create a Task
def create_task(
    title: str,
    duration_hours: float = 1.0,
    due_days: int = 7,
    priority: TaskPriority = TaskPriority.P3,
    notes: str = "",
    category: EventCategory = EventCategory.WORK,
    min_work_hours: float = 0.25,
    max_work_hours: float = None,
) -> Task:
    """Create a new task with sensible defaults."""
    task = Task(
        title=title,
        notes=notes,
        priority=priority,
        event_category=category,
        due=datetime.now() + timedelta(days=due_days),
    )
    
    task.duration = duration_hours
    task.min_work_duration = min_work_hours
    task.max_work_duration = max_work_hours or duration_hours
    
    task.save()
    print(f"Created task: {task.title} (ID: {task.id})")
    return task

# Example (uncomment to use):
# new_task = create_task(
#     title="Test Task from Notebook",
#     duration_hours=2.0,
#     due_days=5,
#     priority=TaskPriority.P2,
#     notes="Created for testing"
# )

In [ ]:
# Cell 15: Get Task by ID
def explore_task(task_id: int) -> Task:
    """Get detailed information about a specific task."""
    task = Task.get(task_id)
    
    print(f"=== {task.title} ===")
    print(f"ID: {task.id}")
    print(f"Status: {task.status.value if task.status else 'UNKNOWN'}")
    print(f"Priority: {task.priority.value if task.priority else '?'}")
    print(f"Category: {task.event_category.value if task.event_category else '?'}")
    print(f"Due: {task.due.strftime('%Y-%m-%d %H:%M') if task.due else 'Not set'}")
    print(f"\nTime:")
    print(f"  Duration: {task.duration} hours" if task.duration else "  Duration: Not set")
    print(f"  Min chunk: {task.min_work_duration} hours" if task.min_work_duration else "  Min chunk: Not set")
    print(f"  Max chunk: {task.max_work_duration} hours" if task.max_work_duration else "  Max chunk: Not set")
    print(f"  Spent: {task.time_chunks_spent / 4 if task.time_chunks_spent else 0} hours")
    print(f"  Remaining: {task.time_chunks_remaining / 4 if task.time_chunks_remaining else '?'} hours")
    print(f"\nFlags:")
    print(f"  On Deck (Up Next): {task.on_deck}")
    print(f"  At Risk: {task.at_risk}")
    print(f"\nNotes: {task.notes or '(none)'}")
    
    return task

# Example
if tasks:
    example_task = explore_task(tasks[0].id)

In [ ]:
# Cell 16: Planner Actions - Start/Stop Work
def start_task(task: Task) -> None:
    """Start working on a task."""
    print(f"Starting: {task.title}")
    task.start()
    print(f"Status: {task.status.value}")

def stop_task(task: Task) -> None:
    """Stop working on a task."""
    print(f"Stopping: {task.title}")
    task.stop()
    print(f"Status: {task.status.value}")

# Example (uncomment to use):
# task = Task.get(123)
# start_task(task)
# # ... do some work ...
# stop_task(task)

In [ ]:
# Cell 17: Log Work
def log_work_on_task(task: Task, minutes: int, end_time: datetime = None) -> None:
    """Log work time on a task."""
    print(f"Logging {minutes} minutes on: {task.title}")
    
    before_spent = task.time_chunks_spent / 4 if task.time_chunks_spent else 0
    
    task.log_work(minutes, end=end_time or datetime.now())
    task.refresh()
    
    after_spent = task.time_chunks_spent / 4 if task.time_chunks_spent else 0
    print(f"Time spent: {before_spent}h -> {after_spent}h")

# Example (uncomment to use):
# task = Task.get(123)
# log_work_on_task(task, minutes=45)

In [ ]:
# Cell 18: Add Time to Estimate
def add_time_to_task(task: Task, hours: float) -> None:
    """Add time to a task's total estimate."""
    print(f"Adding {hours} hours to: {task.title}")
    
    before_duration = task.duration
    task.add_time(hours)
    task.refresh()
    after_duration = task.duration
    
    print(f"Duration: {before_duration}h -> {after_duration}h")

# Example (uncomment to use):
# task = Task.get(123)
# add_time_to_task(task, 0.5)  # Add 30 minutes

In [ ]:
# Cell 19: Mark Complete/Incomplete
def complete_task(task: Task) -> None:
    """Mark a task as complete."""
    print(f"Completing: {task.title}")
    task.mark_complete()
    print(f"Status: {task.status.value}")

def uncomplete_task(task: Task) -> None:
    """Mark a task as incomplete (unarchive)."""
    print(f"Uncompleting: {task.title}")
    task.mark_incomplete()
    print(f"Status: {task.status.value}")

# Example (uncomment to use):
# task = Task.get(123)
# complete_task(task)
# # Oops, not done yet:
# uncomplete_task(task)

In [ ]:
# Cell 20: Filter Tasks
def filter_tasks(
    status: Optional[TaskStatus] = None,
    priority: Optional[TaskPriority] = None,
    category: Optional[EventCategory] = None,
    overdue: bool = None,
    at_risk: bool = None,
    on_deck: bool = None,
    has_due_date: bool = None,
    exclude_done: bool = True,  # By default exclude COMPLETE and ARCHIVED
) -> List[Task]:
    """Filter tasks by multiple criteria.
    
    Note: Both COMPLETE and ARCHIVED statuses represent "done" tasks in Reclaim.
    Set exclude_done=True (default) to filter out both.
    """
    tasks = Task.list()
    
    # Filter out done tasks (COMPLETE and ARCHIVED) unless specifically requested
    if exclude_done and status is None:
        tasks = [t for t in tasks if t.status not in (TaskStatus.COMPLETE, TaskStatus.ARCHIVED)]
    
    if status:
        tasks = [t for t in tasks if t.status == status]
    if priority:
        tasks = [t for t in tasks if t.priority == priority]
    if category:
        tasks = [t for t in tasks if t.event_category == category]
    if overdue is not None:
        now = datetime.now(timezone.utc)
        if overdue:
            tasks = [t for t in tasks if t.due and t.due < now]
        else:
            tasks = [t for t in tasks if not t.due or t.due >= now]
    if at_risk is not None:
        tasks = [t for t in tasks if t.at_risk == at_risk]
    if on_deck is not None:
        tasks = [t for t in tasks if t.on_deck == on_deck]
    if has_due_date is not None:
        if has_due_date:
            tasks = [t for t in tasks if t.due]
        else:
            tasks = [t for t in tasks if not t.due]
    
    return tasks

# Examples
scheduled_tasks = filter_tasks(status=TaskStatus.SCHEDULED)
high_priority = filter_tasks(priority=TaskPriority.P1)
at_risk_tasks = filter_tasks(at_risk=True)

print(f"Scheduled: {len(scheduled_tasks)}")
print(f"High Priority (P1): {len(high_priority)}")
print(f"At Risk: {len(at_risk_tasks)}")

### Task Summary

| Action | Method | Notes |
|--------|--------|-------|
| List all | `Task.list()` | Returns all tasks |
| Get by ID | `Task.get(task_id)` | Fetch specific task |
| Create | `Task(...); task.save()` | Set duration via property |
| Update | `task.property = ...; task.save()` | Persists to server |
| Delete | `task.delete()` | Permanent removal |
| Start work | `task.start()` | Begin working |
| Stop work | `task.stop()` | Pause working |
| Log work | `task.log_work(minutes, end=datetime)` | Record time spent |
| Add time | `task.add_time(hours)` | Increase estimate |
| Complete | `task.mark_complete()` | Mark done |
| Uncomplete | `task.mark_incomplete()` | Reopen task |

#### Time Chunk Conversion
- Reclaim uses 15-minute chunks internally
- `duration` property: hours (e.g., 1.5 = 6 chunks)
- `time_chunks_required`: raw chunks

---
## Section 4: Analytics and Reports

In [ ]:
# Cell 21: Habit Completion Analysis
def habit_completion_analysis():
    """Analyze habit completion rates from periods."""
    habits = SmartHabit.list()
    
    analysis = []
    for h in habits:
        if not h.periods:
            continue
        
        total = len(h.periods)
        completed = sum(1 for p in h.periods if p.done)
        scheduled = sum(1 for p in h.periods if p.event_start)
        locked = sum(1 for p in h.periods if p.locked)
        
        analysis.append({
            "Habit": h.title[:30],
            "Total Periods": total,
            "Completed": completed,
            "Completion %": f"{(completed/total*100):.0f}%" if total > 0 else "N/A",
            "Scheduled": scheduled,
            "Locked": locked,
        })
    
    df = pd.DataFrame(analysis)
    display(df.sort_values("Total Periods", ascending=False))
    return df

completion_df = habit_completion_analysis()

In [ ]:
# Cell 22: Task Status Summary
def task_status_summary():
    """Summarize tasks by status, priority, and time."""
    tasks = Task.list()
    
    # By status
    print("=== By Status ===")
    status_counts = {}
    for t in tasks:
        status = t.status.value if t.status else "UNKNOWN"
        status_counts[status] = status_counts.get(status, 0) + 1
    
    for status, count in sorted(status_counts.items()):
        print(f"  {status}: {count}")
    
    # By priority (incomplete only)
    print("\n=== By Priority (Incomplete) ===")
    priority_counts = {}
    for t in tasks:
        if t.status == TaskStatus.COMPLETE:
            continue
        priority = t.priority.value if t.priority else "?"
        priority_counts[priority] = priority_counts.get(priority, 0) + 1
    
    for priority in ["P1", "P2", "P3", "P4", "?"]:
        if priority in priority_counts:
            print(f"  {priority}: {priority_counts[priority]}")
    
    # Time summary
    print("\n=== Time Summary ===")
    incomplete = [t for t in tasks if t.status != TaskStatus.COMPLETE]
    total_duration = sum(t.duration or 0 for t in incomplete)
    total_spent = sum((t.time_chunks_spent or 0) / 4 for t in tasks)
    print(f"  Total estimated (incomplete): {total_duration:.1f} hours")
    print(f"  Total time spent (all): {total_spent:.1f} hours")
    
    return tasks

task_status_summary()

In [ ]:
# Cell 23: At-Risk and Overdue Report
def risk_report():
    """Report on at-risk and overdue tasks."""
    tasks = Task.list()
    now = datetime.now(timezone.utc)
    
    # At risk tasks - exclude COMPLETE and ARCHIVED (both are done states)
    at_risk = [t for t in tasks if t.at_risk and t.status not in (TaskStatus.COMPLETE, TaskStatus.ARCHIVED)]
    
    # Overdue tasks - exclude COMPLETE and ARCHIVED
    overdue = []
    for t in tasks:
        if t.due and t.status not in (TaskStatus.COMPLETE, TaskStatus.ARCHIVED):
            # Handle timezone
            due = t.due if t.due.tzinfo else t.due.replace(tzinfo=timezone.utc)
            if due < now:
                overdue.append(t)
    
    print(f"=== At-Risk Tasks ({len(at_risk)}) ===")
    for t in at_risk:
        due_str = t.due.strftime("%m/%d") if t.due else "No due"
        priority = t.priority.value if t.priority else "?"
        print(f"  [{priority}] {t.title[:40]} - Due: {due_str}")
    
    print(f"\n=== Overdue Tasks ({len(overdue)}) ===")
    for t in overdue:
        due = t.due if t.due.tzinfo else t.due.replace(tzinfo=timezone.utc)
        days_overdue = (now - due).days
        priority = t.priority.value if t.priority else "?"
        print(f"  [{priority}] {t.title[:40]} - {days_overdue} days overdue")
    
    return at_risk, overdue

at_risk, overdue = risk_report()

In [ ]:
# Cell 24: Changelog Analysis
def changelog_analysis(days: int = 7):
    """Analyze habit rescheduling patterns."""
    all_changelogs = SmartHabit.get_all_changelogs(limit=200)
    
    cutoff = datetime.now(timezone.utc) - timedelta(days=days)
    recent = [e for e in all_changelogs if e.changed_at and e.changed_at > cutoff]
    
    print(f"=== Changelog Analysis (Last {days} days) ===")
    print(f"Total changes: {len(recent)}\n")
    
    # Count by reason
    by_reason = {}
    for entry in recent:
        reason = entry.reason.value.replace("SMART_SERIES_EVENT_", "")
        by_reason[reason] = by_reason.get(reason, 0) + 1
    
    print("By Reason:")
    for reason, count in sorted(by_reason.items(), key=lambda x: -x[1]):
        print(f"  {reason}: {count}")
    
    # Count moves
    moves = [e for e in recent if e.is_move]
    print(f"\nTotal moves: {len(moves)}")
    
    return recent

recent_changes = changelog_analysis(days=7)

---
## Section 5: Export Functions

In [ ]:
# Cell 25: Export Habits to Obsidian
def export_habits_to_obsidian(output_path: str = None) -> str:
    """Export habits to Obsidian-compatible markdown."""
    from pathlib import Path
    habits = SmartHabit.list()
    
    lines = [
        "# Reclaim Smart Habits",
        f"*Exported: {datetime.now().strftime('%Y-%m-%d %H:%M')}*",
        "",
        "## Active Habits",
        "",
    ]
    
    active = [h for h in habits if h.is_enabled]
    for h in sorted(active, key=lambda x: x.title):
        lines.append(f"### {h.title}")
        lines.append(f"- **ID**: `{h.lineage_id}`")
        lines.append(f"- **Type**: {h.event_type.value} ({h.category.value})")
        lines.append(f"- **Duration**: {h.duration_min}-{h.duration_max} minutes")
        lines.append(f"- **Priority**: {h.priority}")
        if h.description:
            lines.append(f"- **Description**: {h.description}")
        if h.recurrence:
            days = ", ".join(h.recurrence.ideal_days)
            lines.append(f"- **Schedule**: {h.recurrence.frequency.value} on {days}")
        lines.append("")
    
    lines.append("## Disabled Habits")
    lines.append("")
    disabled = [h for h in habits if not h.is_enabled]
    for h in sorted(disabled, key=lambda x: x.title):
        lines.append(f"- {h.title} (`{h.lineage_id}`)")
    
    content = "\n".join(lines)
    
    if output_path:
        path = Path(output_path).expanduser()
        path.write_text(content)
        print(f"Exported to {path}")
    
    return content

# Preview (doesn't save)
preview = export_habits_to_obsidian()
print(preview[:1000] + "...")

# To save to Obsidian:
# export_habits_to_obsidian("~/obsidian/Main/Reclaim Habits.md")

In [ ]:
# Cell 26: Export Tasks to Obsidian
def export_tasks_to_obsidian(output_path: str = None, include_completed: bool = False) -> str:
    """Export tasks to Obsidian-compatible markdown."""
    from pathlib import Path
    tasks = Task.list()
    
    if not include_completed:
        tasks = [t for t in tasks if t.status != TaskStatus.COMPLETE]
    
    lines = [
        "# Reclaim Tasks",
        f"*Exported: {datetime.now().strftime('%Y-%m-%d %H:%M')}*",
        "",
    ]
    
    # Group by status
    by_status = {}
    for t in tasks:
        status = t.status.value if t.status else "UNKNOWN"
        by_status.setdefault(status, []).append(t)
    
    for status in ["IN_PROGRESS", "SCHEDULED", "NEW", "COMPLETE"]:
        if status not in by_status:
            continue
        
        lines.append(f"## {status.replace('_', ' ').title()}")
        lines.append("")
        
        for t in by_status[status]:
            checkbox = "x" if t.status == TaskStatus.COMPLETE else " "
            due_str = f" (Due: {t.due.strftime('%Y-%m-%d')})" if t.due else ""
            priority_str = f"[{t.priority.value}]" if t.priority else ""
            lines.append(f"- [{checkbox}] {priority_str} {t.title}{due_str}")
        
        lines.append("")
    
    content = "\n".join(lines)
    
    if output_path:
        path = Path(output_path).expanduser()
        path.write_text(content)
        print(f"Exported to {path}")
    
    return content

# Preview
preview = export_tasks_to_obsidian()
print(preview[:1000] + "...")

# To save:
# export_tasks_to_obsidian("~/obsidian/Main/Reclaim Tasks.md")

In [ ]:
# Cell 27: Export to CSV
def export_to_csv():
    """Export habits and tasks to CSV files."""
    # Habits
    habits = SmartHabit.list()
    habits_data = []
    for h in habits:
        habits_data.append({
            "lineage_id": h.lineage_id,
            "title": h.title,
            "status": h.status.value,
            "event_type": h.event_type.value,
            "category": h.category.value,
            "duration_min": h.duration_min,
            "duration_max": h.duration_max,
            "priority": h.priority,
            "period_count": h.instance_count,
        })
    habits_df = pd.DataFrame(habits_data)
    
    # Tasks
    tasks = Task.list()
    tasks_data = []
    for t in tasks:
        tasks_data.append({
            "id": t.id,
            "title": t.title,
            "status": t.status.value if t.status else "",
            "priority": t.priority.value if t.priority else "",
            "category": t.event_category.value if t.event_category else "",
            "due": t.due.isoformat() if t.due else "",
            "duration_hours": t.duration or "",
            "time_spent_hours": t.time_chunks_spent / 4 if t.time_chunks_spent else 0,
            "on_deck": t.on_deck,
            "at_risk": t.at_risk,
        })
    tasks_df = pd.DataFrame(tasks_data)
    
    return habits_df, tasks_df

habits_csv, tasks_csv = export_to_csv()
print("Habits:")
display(habits_csv.head())
print("\nTasks:")
display(tasks_csv.head())

# To save:
# habits_csv.to_csv("reclaim_habits.csv", index=False)
# tasks_csv.to_csv("reclaim_tasks.csv", index=False)

---
## Section 6: Recipe Appendix

Copy-paste snippets for common operations.

In [ ]:
# RECIPE: List all enabled work habits
habits = SmartHabit.list()
work_enabled = [h for h in habits if h.is_enabled and h.category.value == "WORK"]
for h in work_enabled:
    print(f"- {h.title}")

In [ ]:
# RECIPE: Quick task creation
# task = Task(title="My Task", priority=TaskPriority.P2)
# task.duration = 2.0  # 2 hours
# task.due = datetime.now() + timedelta(days=7)
# task.save()
# print(f"Created: {task.id}")

In [ ]:
# RECIPE: Get weekly time report
tasks = Task.list()
total_hours = sum((t.time_chunks_spent or 0) / 4 for t in tasks)
print(f"Total time logged: {total_hours:.1f} hours")

In [ ]:
# RECIPE: Find most rescheduled habits
changelogs = SmartHabit.get_all_changelogs(limit=100)
cutoff = datetime.now(timezone.utc) - timedelta(days=7)
recent_moves = [e for e in changelogs if e.changed_at > cutoff and e.is_move]

by_series = {}
for entry in recent_moves:
    by_series[entry.series_id] = by_series.get(entry.series_id, 0) + 1

habits = SmartHabit.list()
id_to_title = {h.lineage_id: h.title for h in habits}

print("Most rescheduled this week:")
for sid, count in sorted(by_series.items(), key=lambda x: -x[1])[:5]:
    title = id_to_title.get(sid, f"ID:{sid}")
    print(f"  {title}: {count} times")

In [ ]:
# RECIPE: Batch disable habits containing keyword
# keyword = "example"
# habits = SmartHabit.list()
# for h in habits:
#     if keyword.lower() in h.title.lower() and h.is_enabled:
#         h.disable()
#         print(f"Disabled: {h.title}")

---
## End of Notebook

For more information:
- SDK source: `/Users/raymondyee/C/src/reclaim-sdk`
- API token: https://app.reclaim.ai/settings/developer
- Utility library: `from reclaim_sdk.utils import *`